### DCGAN 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from tqdm import tqdm
import torchvision.datasets as datasets
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
import torchvision.utils as vutils
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch.utils.data import Subset
import numpy as np
from PIL import Image
import os

In [2]:
# Load Image => transform => dataset of all images
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path=root_dir_path
        self.transformations=transformations

        # List of Path of all images
        self.all_img_paths=[os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self,idx):
        img_path=self.all_img_paths[idx]
        img=Image.open(img_path).convert("RGB")

        if self.transformations:
            img=self.transformations(img)

        return img


In [3]:
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

In [4]:
print(f"Device is {device}")

Device is cuda


In [9]:
LEARNING_RATE = 2e-4 
BATCH_SIZE = 128
IMAGE_SIZE = 64
CHANNELS_IMG = 3
Z_DIM = 100
NUM_EPOCHS = 5
FEATURES_DISC = 64
FEATURES_GEN = 64

In [10]:
transformations=transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ]
)

In [6]:
root_dir_path="./img_align_celeba"

In [7]:
dataset=ImageProcessor(root_dir_path,transformations)
dataloader=DataLoader(dataset=dataset,batch_size=128,shuffle=True)

### Generator Network 

In [12]:
class Generator(nn.Module):
    def __init__(self,z_dim,channels_img,features_g):
        super(Generator,self).__init__()

        self.net=nn.Sequential(
            self._block(z_dim,features_g*16,4,1,0), # Input channels=100,Output channels=1024,kernel_size=4,stride=1,padding=0
            
            self.block(features_g*16,features_g*8,4,2,1), # 1024,512,4,2,1

            self.block(features_g*8,features_g*4,4,2,1), # 512,256,4,2,1

            self.block(features_g*4,features_g*2,4,2,1), #256,128,4,2,1

            nn.ConvTranspose2d(
                features_g*2,channels_img,kernel_size=4,stride=2,padding=1
            ),

            nn.Tanh()
        )

    def _block(self,in_channels,out_channels,kernel_size,stride,padding):
        return nn.Sequential(
            nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self,x):
        return self.net(x)

### Discriminator Network

In [14]:
class Discriminator:
    def __init__(self,channels_img,features_d):
        super(Discriminator,self).__init__()

        self.disc=nn.Sequential(
            nn.Conv2d(channels_img,features_d,kernel_size=4,stride=2,padding=1),
            nn.LeakyReLU(0.2),
            self._block(features_d,features_d*2,4,2,1),
            self.block(features_d*2,features_d*4,4,2,1),
            self._block(features_d*4,features_d*8,4,2,1),
            nn.Conv2d(features_d*8,1,kernel_size=4,stride=2,padding=0),
            nn.Sigmoid()
        )

    def _block(self,in_channels,out_channels,kernel_size,stride,padding):
        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2)
        )

    def forward(self,x):
        return self.disc(x)
